In [1]:
import pandas as pd
import numpy as np

import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

In [2]:
input_file = "./OriginData/水位数据20241015.xlsx"
out_file = "./OutputData/merged_涌水量水位.xlsx"

In [3]:
# 读取Excel文件中的所有sheets
datas = pd.read_excel(input_file, sheet_name=None)

In [4]:
datas

{'SBZK 670-118-1':            位置点 Unnamed: 1 Unnamed: 2             时间是具体时间点 Unnamed: 4
 0     横坐标（经纬度）        纵坐标  高程（终孔，井底）                   案例      水位（m)
 1    3045453.7   401240.1      671.2  2022-12-21 00:00:00        622
 2    3045453.7   401240.1      671.2  2022-12-22 00:00:00        NaN
 3    3045453.7   401240.1      671.2  2022-12-23 00:00:00        NaN
 4    3045453.7   401240.1      671.2  2022-12-24 00:00:00        NaN
 ..         ...        ...        ...                  ...        ...
 504  3045453.7   401240.1      671.2  2024-05-07 00:00:00    470.394
 505  3045453.7   401240.1      671.2  2024-05-08 00:00:00    470.046
 506  3045453.7   401240.1      671.2  2024-05-09 00:00:00    469.835
 507  3045453.7   401240.1      671.2  2024-05-10 00:00:00    469.534
 508  3045453.7   401240.1      671.2  2024-05-11 00:00:00    469.154
 
 [509 rows x 5 columns],
 'SBZK670-110-1':           位置点 Unnamed: 1 Unnamed: 2             时间是具体时间点 Unnamed: 4
 0    横坐标（经纬度）        纵坐标  高程

In [5]:
# 将所有sheets合并成一个DataFrame
df = pd.concat(datas.values())
df = df.drop(df.columns[-1], axis=1)
df = df.drop(0, axis=0)
df.columns = ["经度","纬度","高程","日期","水位"]
# 1. 将日期列转换为 datetime 类型，无法解析的设为 NaT
df['日期'] = pd.to_datetime(df['日期'], errors='coerce')  
# 2. 删除日期为 NaT（不规范）的行
df = df.dropna(subset=['日期'])  
df['水位'] = pd.to_numeric(df['水位'], errors='coerce')
df['经度'] = pd.to_numeric(df['经度'], errors='coerce')
df['纬度'] = pd.to_numeric(df['纬度'], errors='coerce')
df['高程'] = pd.to_numeric(df['高程'], errors='coerce')
df = df.dropna(subset=['水位'])  
df = df.dropna(subset=['经度'])
df = df.dropna(subset=['纬度'])
df = df.dropna(subset=['高程']) 
df.dropna()

,经度,纬度,高程,日期,水位
1,3045453.700,4.012401e+05,671.200,2022-12-21,622.000
8,3045453.700,4.012401e+05,671.200,2022-12-28,618.700
14,3045453.700,4.012401e+05,671.200,2023-01-03,614.500
20,3045453.700,4.012401e+05,671.200,2023-01-09,611.600
29,3045453.700,4.012401e+05,671.200,2023-01-18,610.000
...,...,...,...,...,...
799,3044799.788,3.540080e+07,904.286,2024-05-07,753.971
800,3044799.788,3.540080e+07,904.286,2024-05-08,753.967
801,3044799.788,3.540080e+07,904.286,2024-05-09,753.830
802,3044799.788,3.540080e+07,904.286,2024-05-10,753.278


In [6]:
df

,经度,纬度,高程,日期,水位
1,3045453.700,4.012401e+05,671.200,2022-12-21,622.000
8,3045453.700,4.012401e+05,671.200,2022-12-28,618.700
14,3045453.700,4.012401e+05,671.200,2023-01-03,614.500
20,3045453.700,4.012401e+05,671.200,2023-01-09,611.600
29,3045453.700,4.012401e+05,671.200,2023-01-18,610.000
...,...,...,...,...,...
799,3044799.788,3.540080e+07,904.286,2024-05-07,753.971
800,3044799.788,3.540080e+07,904.286,2024-05-08,753.967
801,3044799.788,3.540080e+07,904.286,2024-05-09,753.830
802,3044799.788,3.540080e+07,904.286,2024-05-10,753.278


In [7]:
# 确保时间列是datetime类型
df['日期'] = pd.to_datetime(df['日期'])
grouped = df.groupby(['经度', '纬度', '高程'])
# 现在grouped是一个GroupBy对象，你可以遍历它，或者对每个组进行操作
df_list = []
for name, group in grouped:
    print("group type : ",type(group))
    # 首先将'日期'列设置为索引
    group.set_index('日期', inplace=True)
    
    # 确保时间序列是完整的，使用resample方法来按所需的时间频率填充缺失的时间点
    df_resampled = group.resample('D').mean()
    
    # 使用线性插值来填充缺失的数据
    df_interpolated = df_resampled.interpolate(method='linear')
    
    # 索引重置为列
    df_interpolated.reset_index(inplace=True)

    #将插值后的数据加入
    df_list.append(df_interpolated)

    print(f"Group: {name}")
    print(df_interpolated)

# 使用pd.concat将列表中的所有DataFrame合并成一个单一的DataFrame
df_linear = pd.concat(df_list, ignore_index=True)
df_linear = df_linear.round(3)

group type :  <class 'pandas.core.frame.DataFrame'>
Group: (3044699.547, 401076.749, 905.05)
            日期           经度          纬度      高程       水位
0   2023-03-31  3044699.547  401076.749  905.05  865.482
1   2023-04-01  3044699.547  401076.749  905.05  866.392
2   2023-04-02  3044699.547  401076.749  905.05  867.302
3   2023-04-03  3044699.547  401076.749  905.05  868.212
4   2023-04-04  3044699.547  401076.749  905.05  869.122
..         ...          ...         ...     ...      ...
403 2024-05-07  3044699.547  401076.749  905.05  853.677
404 2024-05-08  3044699.547  401076.749  905.05  853.595
405 2024-05-09  3044699.547  401076.749  905.05  853.582
406 2024-05-10  3044699.547  401076.749  905.05  853.625
407 2024-05-11  3044699.547  401076.749  905.05  853.476

[408 rows x 5 columns]
group type :  <class 'pandas.core.frame.DataFrame'>
Group: (3044705.773, 400634.262, 902.88)
            日期           经度          纬度      高程       水位
0   2023-02-02  3044705.773  400634.262  902.88  

In [8]:
df_linear

,日期,经度,纬度,高程,水位
0,2023-03-31,3.044700e+06,401076.749,905.05,865.482
1,2023-04-01,3.044700e+06,401076.749,905.05,866.392
2,2023-04-02,3.044700e+06,401076.749,905.05,867.302
3,2023-04-03,3.044700e+06,401076.749,905.05,868.212
4,2023-04-04,3.044700e+06,401076.749,905.05,869.122
...,...,...,...,...,...
15572,2024-05-07,3.045730e+09,398843.511,941.53,909.628
15573,2024-05-08,3.045730e+09,398843.511,941.53,909.774
15574,2024-05-09,3.045730e+09,398843.511,941.53,910.064
15575,2024-05-10,3.045730e+09,398843.511,941.53,910.356


# 制作Web数据集

In [10]:
# 在第0列位置插入"序号"列，初始值为 None
df_linear.insert(0, "序号", None)
# 按索引填充序号值（如 0, 1, 2, 3...）
df_linear.loc[:, "序号"] = df_linear.index  # 通过 loc 安全赋值索引值
df_linear['日期'] = pd.to_datetime(df_linear['日期'])

In [11]:
df_linear.to_excel("./OutputData/water_level.xlsx",index=False)

In [12]:
df_linear

,序号,日期,经度,纬度,高程,水位
0,0,2023-03-31,3.044700e+06,401076.749,905.05,865.482
1,1,2023-04-01,3.044700e+06,401076.749,905.05,866.392
2,2,2023-04-02,3.044700e+06,401076.749,905.05,867.302
3,3,2023-04-03,3.044700e+06,401076.749,905.05,868.212
4,4,2023-04-04,3.044700e+06,401076.749,905.05,869.122
...,...,...,...,...,...,...
15572,15572,2024-05-07,3.045730e+09,398843.511,941.53,909.628
15573,15573,2024-05-08,3.045730e+09,398843.511,941.53,909.774
15574,15574,2024-05-09,3.045730e+09,398843.511,941.53,910.064
15575,15575,2024-05-10,3.045730e+09,398843.511,941.53,910.356


In [13]:
df_linear['日期'].dtype

dtype('<M8[ns]')

In [14]:
df_1 = df_linear.iloc[[0]]

In [15]:
df_1

,序号,日期,经度,纬度,高程,水位
0,0,2023-03-31,3044699.547,401076.749,905.05,865.482


In [16]:
df_1.to_excel("./OutputData/water_level_1.xlsx",index=False)